# NB_Silver_Closed_Deals — Bronze to Silver Transformation

## Section 1 — Purpose, Business Context & Verified Profile

- **Source Table:** `Cloud_Retail_Data_Lakehouse_v2.dbo.closed_deals` → OneLake Shortcut (`Files/bronze/marketing/closed_deals/`).
- **Silver Target:** `silver.closed_deals` (Managed Delta Table) + `silver._quarantine_closed_deals` (Conditional Quarantine).
- **Grain:** One row per won/closed seller deal (`mql_id` PK / `seller_id` unique, `842` rows).
- **Verified Bronze Profile (`842` rows × `14` columns):**
  - `0` duplicate `mql_id`s, `0` duplicate `seller_id`s, `0` invalid 32-char hex IDs across `mql_id`, `seller_id`, `sdr_id` (`32` SDRs), and `sr_id` (`22` SRs).
  - **MQL Referential Integrity:** `0` orphan `mql_id`s vs `silver.marketing_qualified_leads` (`842 / 8,000 = 10.53%` conversion).
  - **Store Seller Activity (`silver.sellers`):** Only **`380`** of `842` won sellers exist in `silver.sellers` (`is_active_seller_in_store = True`); the remaining **`462`** sellers have no store orders yet (`False`).
  - **Temporal Anomaly:** **`1`** row (`mql_id = b91cf8812365f50ff4bda4bcd6206b05`) has `won_date` (`2018-03-06 19:38:55`) two days before `first_contact_date` (`2018-03-08`) → flagged via `has_won_before_first_contact = True` (never dropped).
  - **Type Traps Handled:** `average_stock` is a categorical range string (`'1-5'`, `'5-20'`, `'20-50'`, `'50-200'`, `'200+'`, `'unknown'`), `lead_behaviour_profile` contains multi-label values (e.g., `'cat, wolf'`), and `declared_monthly_revenue` reaches `50,000,000.00` (`DECIMAL(14,2)`).

> **WARNING — SAFETY LOCK ENABLED (`WRITE_ENABLED = False`):** Do not enable writes until Dry Run validation passes.

## Section 2 — Configuration & Snapshot Auto-Discovery

Auto-discovers the latest `.parquet` snapshot inside `Files/bronze/marketing/closed_deals/`, sets canonical Silver targets, and enables cross-table enrichment against `silver.marketing_qualified_leads` and `silver.sellers`.

In [ ]:
# Section 2: Configuration & Snapshot Auto-Discovery
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

WRITE_ENABLED = False  # SAFETY LOCK: flip to True only after Dry Run review
RUN_CROSS_TABLE_CHECKS = True

BRONZE_FOLDER = "Files/bronze/marketing/closed_deals"
parquet_files = sorted([f.name for f in notebookutils.fs.ls(BRONZE_FOLDER) if f.name.endswith(".parquet")])
if not parquet_files:
    raise FileNotFoundError(f"No Parquet snapshots found in {BRONZE_FOLDER}")

LATEST_FILE = parquet_files[-1]
BRONZE_PATH = f"{BRONZE_FOLDER}/{LATEST_FILE}"
INGESTION_RUN_ID = LATEST_FILE.rsplit("_", 1)[-1].replace(".parquet", "") if "_" in LATEST_FILE else "manual_run"

SILVER_SCHEMA = "silver"
SILVER_TARGET_TABLE = f"{SILVER_SCHEMA}.closed_deals"
QUARANTINE_TARGET_TABLE = f"{SILVER_SCHEMA}._quarantine_closed_deals"
SILVER_MQL_TABLE = f"{SILVER_SCHEMA}.marketing_qualified_leads"
SILVER_SELLERS_TABLE = f"{SILVER_SCHEMA}.sellers"

SOURCE_SYSTEM = "sqlserver_olist"
SOURCE_TABLE = "dbo.closed_deals"

EXPECTED_BRONZE_COLS = [
    "mql_id",
    "seller_id",
    "sdr_id",
    "sr_id",
    "won_date",
    "business_segment",
    "lead_type",
    "lead_behaviour_profile",
    "has_company",
    "has_gtin",
    "average_stock",
    "business_type",
    "declared_product_catalog_size",
    "declared_monthly_revenue",
]

print(f"Discovered Parquet snapshots : {len(parquet_files)}")
print(f"Selected BRONZE_PATH         : {BRONZE_PATH}")
print(f"Extracted INGESTION_RUN_ID   : {INGESTION_RUN_ID}")
print(f"Configured SILVER_TARGET     : {SILVER_TARGET_TABLE}")
print(f"WRITE_ENABLED                : {WRITE_ENABLED}")

## Section 3 — Distributed Bronze Parquet Ingestion

Reads the discovered Bronze Parquet snapshot directly with PySpark and attaches `_source_file` lineage.

In [ ]:
# Section 3: Read Bronze Parquet & Inspect Schema
df_bronze_raw = (
    spark.read
    .format("parquet")
    .load(BRONZE_PATH)
    .withColumn("_source_file", F.input_file_name())
)

bronze_count = df_bronze_raw.count()
print(f"Bronze Closed Deals Row Count: {bronze_count:,} (Expected: 842)")
df_bronze_raw.printSchema()
display(df_bronze_raw.limit(5))

## Stage 1 — Schema Contract & Raw Null Baseline

Verifies that all `14` expected Bronze columns exist, checks for schema drift, and prints raw `NULL` counts.

In [ ]:
# Stage 1: Schema Contract & Baseline Profiling
actual_data_cols = [c for c in df_bronze_raw.columns if c != "_source_file"]
missing_cols = [c for c in EXPECTED_BRONZE_COLS if c not in actual_data_cols]
extra_cols = [c for c in actual_data_cols if c not in EXPECTED_BRONZE_COLS]

if missing_cols:
    raise ValueError(f"Stage 1 FAILED — Missing required Bronze columns: {missing_cols}")

if extra_cols:
    print(f"WARNING — Extra columns observed in Bronze (excluded from Silver contract): {extra_cols}")

df_bronze = df_bronze_raw.select(*EXPECTED_BRONZE_COLS, "_source_file")

raw_nulls = df_bronze.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"{c}_nulls")
    for c in EXPECTED_BRONZE_COLS
])
print("Stage 1 Schema Validation: PASSED")
display(raw_nulls)

## Stage 2 — Standardization & Safe Type Casting

1. Lowercases and trims all 4 hex keys (`mql_id`, `seller_id`, `sdr_id`, `sr_id`) and categorical text fields (`business_segment`, `lead_type`, `lead_behaviour_profile`, `average_stock`, `business_type`).
2. Safe-casts `won_date` (`TIMESTAMP`), `has_company`/`has_gtin` (`BOOLEAN`), `declared_product_catalog_size` (`INT`), and `declared_monthly_revenue` (`DECIMAL(14,2)`).

In [ ]:
# Stage 2: Standardization & Safe Type Casting
def clean_opt_str(col_name):
    trimmed = F.lower(F.trim(F.col(col_name).cast("string")))
    return F.when(F.length(trimmed) == 0, F.lit(None).cast("string")).otherwise(trimmed)

df_pre = (
    df_bronze
    .withColumn("_raw_won_str", F.col("won_date").cast("string"))
    .withColumn("_raw_rev_str", F.col("declared_monthly_revenue").cast("string"))
)

df_std = (
    df_pre
    .withColumn("mql_id", F.lower(F.trim(F.col("mql_id").cast("string"))))
    .withColumn("seller_id", F.lower(F.trim(F.col("seller_id").cast("string"))))
    .withColumn("sdr_id", F.lower(F.trim(F.col("sdr_id").cast("string"))))
    .withColumn("sr_id", F.lower(F.trim(F.col("sr_id").cast("string"))))
    .withColumn("won_date", F.expr("try_cast(won_date AS TIMESTAMP)"))
    .withColumn("business_segment", clean_opt_str("business_segment"))
    .withColumn("lead_type", clean_opt_str("lead_type"))
    .withColumn("lead_behaviour_profile", clean_opt_str("lead_behaviour_profile"))
    .withColumn("has_company", F.expr("try_cast(has_company AS BOOLEAN)"))
    .withColumn("has_gtin", F.expr("try_cast(has_gtin AS BOOLEAN)"))
    .withColumn("average_stock", clean_opt_str("average_stock"))
    .withColumn("business_type", clean_opt_str("business_type"))
    .withColumn("declared_product_catalog_size", F.expr("try_cast(declared_product_catalog_size AS INT)"))
    .withColumn("declared_monthly_revenue", F.expr("try_cast(declared_monthly_revenue AS DECIMAL(14,2))"))
    .withColumn("_cast_failed", (
        (F.col("_raw_won_str").isNotNull() & F.col("won_date").isNull()) |
        (F.col("_raw_rev_str").isNotNull() & F.col("declared_monthly_revenue").isNull())
    ))
)

print(f"Stage 2 complete. Cast failures: {df_std.filter(F.col('_cast_failed')).count()}")

## Stage 3 & 4 — PK/Unique Key Validation, MQL FK Check & Quarantine Routing

1. Enforces uniqueness on `mql_id` (PK) and `seller_id`, plus `^[0-9a-f]{32}$` on all 4 hex IDs and `declared_monthly_revenue >= 0`.
2. Checks `mql_id` Referential Integrity against `silver.marketing_qualified_leads` (`0` orphans expected) and routes any invalid record to `df_quarantine`.

In [ ]:
# Stage 3 & 4: Key Validation, MQL FK Check & Quarantine Routing
hex32_regex = r"^[0-9a-f]{32}$"
exact_dup_count = bronze_count - df_bronze.dropDuplicates(EXPECTED_BRONZE_COLS).count()

dup_mql_df = (
    df_std.filter(F.col("mql_id").isNotNull())
    .groupBy("mql_id").count().filter(F.col("count") > 1)
    .select("mql_id").withColumn("_dup_mql", F.lit(True))
)
dup_seller_cnt = df_std.filter(F.col("seller_id").isNotNull()).groupBy("seller_id").count().filter(F.col("count") > 1).count()

if RUN_CROSS_TABLE_CHECKS and spark.catalog.tableExists(SILVER_MQL_TABLE):
    mql_ref = spark.table(SILVER_MQL_TABLE).select("mql_id", "first_contact_date").distinct()
    missing_mql_df = (
        df_std.select("mql_id").distinct()
        .join(mql_ref.select("mql_id"), on="mql_id", how="left_anti")
        .withColumn("_missing_mql_fk", F.lit(True))
    )
    missing_mql_cnt = missing_mql_df.count()
    print(f"Referential Check vs {SILVER_MQL_TABLE}: {missing_mql_cnt} orphan mql_id(s) (Expected: 0)")
else:
    missing_mql_df = spark.createDataFrame([], "mql_id string, _missing_mql_fk boolean")
    print(f"Referential Check vs {SILVER_MQL_TABLE} skipped (table not found or disabled).")

df_checked = (
    df_std
    .join(dup_mql_df, on="mql_id", how="left")
    .withColumn("_dup_mql", F.coalesce(F.col("_dup_mql"), F.lit(False)))
    .join(missing_mql_df, on="mql_id", how="left")
    .withColumn("_missing_mql_fk", F.coalesce(F.col("_missing_mql_fk"), F.lit(False)))
    .withColumn(
        "dq_rejection_reason",
        F.concat_ws(
            " | ",
            F.when(~F.coalesce(F.col("mql_id"), F.lit("")).rlike(hex32_regex), F.lit("INVALID_MQL_ID_HEX32")),
            F.when(~F.coalesce(F.col("seller_id"), F.lit("")).rlike(hex32_regex), F.lit("INVALID_SELLER_ID_HEX32")),
            F.when(~F.coalesce(F.col("sdr_id"), F.lit("")).rlike(hex32_regex), F.lit("INVALID_SDR_ID_HEX32")),
            F.when(~F.coalesce(F.col("sr_id"), F.lit("")).rlike(hex32_regex), F.lit("INVALID_SR_ID_HEX32")),
            F.when(F.col("won_date").isNull(), F.lit("NULL_OR_INVALID_WON_DATE")),
            F.when(F.col("declared_monthly_revenue").isNull() | (F.col("declared_monthly_revenue") < 0), F.lit("INVALID_MONTHLY_REVENUE")),
            F.when(F.col("_cast_failed"), F.lit("TYPE_CAST_FAILED")),
            F.when(F.col("_dup_mql"), F.lit("DUPLICATE_MQL_ID_PK")),
            F.when(F.col("_missing_mql_fk"), F.lit("MISSING_MQL_FK")),
        )
    )
)

df_valid = df_checked.filter(F.length(F.col("dq_rejection_reason")) == 0)
df_quarantine = (
    df_checked.filter(F.length(F.col("dq_rejection_reason")) > 0)
    .select(*EXPECTED_BRONZE_COLS, "dq_rejection_reason", "_source_file")
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumn("quarantined_at", F.current_timestamp())
)

valid_count = df_valid.count()
quarantine_count = df_quarantine.count()
print(f"Exact Duplicates : {exact_dup_count} | Duplicate seller_id : {dup_seller_cnt}")
print(f"Valid Deals      : {valid_count:,} (Expected: 842)")
print(f"Quarantined Rows : {quarantine_count:,} (Expected: 0)")

## Stage 5 — Cross-Table Funnel Enrichment, Lineage & Reconciliation

Enriches `df_valid` with `3` high-value analytical columns (using `silver.marketing_qualified_leads` and `silver.sellers` when present):
- `lead_conversion_days`: `DATEDIFF(CAST(won_date AS DATE), first_contact_date)`.
- `has_won_before_first_contact`: `True` for the `1` anomalous row where `won_date < first_contact_date` (`b91cf8812365f50ff4bda4bcd6206b05`).
- `is_active_seller_in_store`: `True` if `seller_id` exists in `silver.sellers` (`380` active sellers vs `462` not yet selling).

In [ ]:
# Stage 5: Cross-Table Enrichment, Lineage & Reconciliation
if RUN_CROSS_TABLE_CHECKS and spark.catalog.tableExists(SILVER_MQL_TABLE):
    mql_dates = spark.table(SILVER_MQL_TABLE).select("mql_id", "first_contact_date").distinct()
    df_enriched = (
        df_valid.join(mql_dates, on="mql_id", how="left")
        .withColumn("lead_conversion_days", F.datediff(F.to_date("won_date"), F.col("first_contact_date")))
        .withColumn("has_won_before_first_contact", F.to_date("won_date") < F.col("first_contact_date"))
        .drop("first_contact_date")
    )
else:
    df_enriched = (
        df_valid
        .withColumn("lead_conversion_days", F.lit(None).cast("int"))
        .withColumn("has_won_before_first_contact", F.lit(None).cast("boolean"))
    )

if RUN_CROSS_TABLE_CHECKS and spark.catalog.tableExists(SILVER_SELLERS_TABLE):
    store_sellers = spark.table(SILVER_SELLERS_TABLE).select("seller_id").distinct().withColumn("_in_store", F.lit(True))
    df_enriched = (
        df_enriched.join(store_sellers, on="seller_id", how="left")
        .withColumn("is_active_seller_in_store", F.coalesce(F.col("_in_store"), F.lit(False)))
        .drop("_in_store")
    )
else:
    df_enriched = df_enriched.withColumn("is_active_seller_in_store", F.lit(None).cast("boolean"))

df_silver = (
    df_enriched
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumn("silver_processed_at", F.current_timestamp())
    .select(
        "mql_id",
        "seller_id",
        "sdr_id",
        "sr_id",
        "won_date",
        "business_segment",
        "lead_type",
        "lead_behaviour_profile",
        "has_company",
        "has_gtin",
        "average_stock",
        "business_type",
        "declared_product_catalog_size",
        "declared_monthly_revenue",
        "lead_conversion_days",
        "has_won_before_first_contact",
        "is_active_seller_in_store",
        "source_system",
        "source_table",
        "ingestion_run_id",
        "silver_processed_at",
    )
)

silver_count = df_silver.count()
assert bronze_count == silver_count + quarantine_count, (
    f"Reconciliation mismatch: bronze={bronze_count}, silver={silver_count}, quarantine={quarantine_count}"
)

diag = df_silver.agg(
    F.sum(F.when(F.col("has_won_before_first_contact") == True, 1).otherwise(0)).alias("won_before_fcd"),
    F.sum(F.when(F.col("is_active_seller_in_store") == True, 1).otherwise(0)).alias("active_in_store"),
    F.sum(F.when(F.col("is_active_seller_in_store") == False, 1).otherwise(0)).alias("inactive_in_store"),
).collect()[0]

print(f"Stage 5 Reconciliation PASSED : bronze={bronze_count:,}, silver={silver_count:,}, quarantine={quarantine_count:,}")
print(f"won_date < first_contact_date : {diag['won_before_fcd']} (Expected: 1)")
print(f"Active vs Inactive in Store   : {diag['active_in_store']} active / {diag['inactive_in_store']} inactive (Expected: 380 / 462)")
df_silver.printSchema()

## Section 5 — Guarded Delta Write (`silver.closed_deals`)

Creates `silver` schema if missing and persists `silver.closed_deals` strictly when `WRITE_ENABLED = True`.

In [ ]:
# Section 5: Guarded Delta Write
if WRITE_ENABLED:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
    df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER_TARGET_TABLE)
    print(f"Wrote {silver_count:,} rows to {SILVER_TARGET_TABLE}")

    if quarantine_count > 0:
        df_quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(QUARANTINE_TARGET_TABLE)
        print(f"Wrote {quarantine_count:,} rows to {QUARANTINE_TARGET_TABLE}")
    else:
        print("Zero quarantined rows — skipping quarantine table write.")
else:
    print("WRITE_ENABLED is False — dry validation only; no Silver or Quarantine data was written.")

## Section 6 — Post-Write Delta Read-Back Verification

Reads back `silver.closed_deals` when `WRITE_ENABLED = True` and verifies `842` rows and `0` duplicate `mql_id`s.

In [ ]:
# Section 6: Post-Write Validation
if WRITE_ENABLED:
    df_check = spark.table(SILVER_TARGET_TABLE)
    chk_count = df_check.count()
    chk_dup = df_check.groupBy("mql_id").count().filter(F.col("count") > 1).count()
    assert chk_count == silver_count, f"Row mismatch: {chk_count} vs {silver_count}"
    assert chk_dup == 0, f"Duplicate mql_id found: {chk_dup}"
    print(f"Post-write validation PASSED — rows={chk_count:,}, dup_pk={chk_dup}")
    display(df_check.limit(5))
else:
    print("Post-write validation skipped because WRITE_ENABLED is False.")